02_eda

In [ ]:
#1
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

OUT_DIR = Path("outputs")
stage1 = pd.read_pickle(OUT_DIR / "stage1_all_women.pkl")
stage2 = pd.read_pickle(OUT_DIR / "stage2_working_women.pkl")
outside = pd.read_pickle(OUT_DIR / "outside_lf_women.pkl")
fl = pd.read_pickle(OUT_DIR / "feature_lists.pkl")
CAT, NUM = list(fl["FEATURES_CAT"]), list(fl["FEATURES_NUM"])
VL = pd.read_pickle(OUT_DIR / "value_labels.pkl")

SRC = {"sector": "b1q3_perv1", "social_group": "b3q4_hhv1", "religion": "b3q3_hhv1",
       "state": "state_perv1", "marital": "b4q7_perv1"}

def lab(series, var):
    """Turn codes into SPSS value labels. '07' -> '7' and '7.0' -> '7', so state codes 01-09 match their labels."""
    m = {}
    for k, v in VL.get(SRC.get(var, var), {}).items():
        try:
            m[str(int(float(k)))] = v
        except Exception:
            m[str(k)] = v
    s = (series.astype(str)
               .str.replace(r"\.0$", "", regex=True)
               .str.replace(r"^0+(?=\d)", "", regex=True))
    return s.map(m).fillna(s)

print(stage1.shape, stage2.shape, outside.shape)

In [ ]:
#2
# ---- 1. Data preview: 8 random women (no household ID), labels instead of codes ----
PREVIEW = ["state", "sector", "age", "marital", "edu_group", "edu_years", "religion", "social_group",
           "hh_size", "log_pc_exp", "hh_kids_u6", "lfp", "own_economy_type"]
preview = stage1[PREVIEW].sample(8, random_state=42).copy()
for c in ["state", "sector", "marital", "religion", "social_group"]:
    preview[c] = lab(preview[c], c)
preview["log_pc_exp"] = preview["log_pc_exp"].round(2)
display(preview.reset_index(drop=True))
preview.reset_index(drop=True).to_csv(OUT_DIR / "data_preview.csv", index=False)

# ---- 2. Variable dictionary ----
dictionary = pd.DataFrame([
    ("lfp", "Target, Stage 1", "1 = working or seeking work (usual status 11, 12, 21, 31, 41, 51, 81), 0 = outside the labour force", "binary"),
    ("y_gig_vs_trad", "Target, Stage 2", "1 = gig-proxy, 0 = traditional, working women only (startup-proxy set aside)", "binary"),
    ("age, age_sq", "Feature", "Age in years and its square", "numeric"),
    ("marital", "Feature", "Marital status (b4q7_perv1)", "categorical"),
    ("edu_group", "Feature", "Education level rebuilt from b4q8_perv1 (8 groups)", "categorical"),
    ("edu_years", "Feature", "Years of formal education (b4q10_perv1)", "numeric"),
    ("sector", "Feature", "Rural or urban (b1q3_perv1)", "binary"),
    ("state", "Feature", "State or union territory (state_perv1), 36 values", "categorical"),
    ("religion", "Feature", "Household religion (b3q3_hhv1)", "categorical"),
    ("social_group", "Feature", "Household social group (b3q4_hhv1)", "categorical"),
    ("hh_size", "Feature", "Household size (b3q1_hhv1)", "numeric"),
    ("log_pc_exp", "Feature", "Log of monthly household consumption per person (b3q5pt6_hhv1 / b3q1_hhv1)", "numeric"),
    ("hh_kids_u6", "Feature", "Household members under 6", "numeric"),
    ("hh_elders_60plus", "Feature", "Household members aged 60 or more", "numeric"),
    ("hh_adult_men", "Feature", "Men aged 15-59 in the household", "numeric"),
    ("hh_max_edu_others", "Feature", "Highest years of education among the other household members", "numeric"),
    ("own_economy_type", "Analysis only", "Gig-proxy, startup-proxy or traditional (from NCO occupation and usual status)", "categorical"),
    ("Total_Income, b6q9_3pt*", "Analysis only", "Earnings used in the Door or Trap section; never used as features", "numeric"),
    ("Social_Security, b5pt1q11_perv1", "Analysis only", "Social security benefits and job contract type; asked of some workers only", "categorical"),
], columns=["variable", "role", "meaning", "type"])
display(dictionary)
dictionary.to_csv(OUT_DIR / "variable_dictionary.csv", index=False)

# ---- 3. Sample flow and train/test split ----
flow = pd.DataFrame([
    ("All persons in the PLFS file", 418_159),
    ("Women aged 15-59 (Stage 1 sample)", len(stage1)),
    ("  in the labour force (lfp = 1)", int(stage1["lfp"].sum())),
    ("    working", int((stage1["lfp_group"] == "working").sum())),
    ("    seeking work", int((stage1["lfp_group"] == "seeking").sum())),
    ("  outside the labour force", int((stage1["lfp_group"] == "outside").sum())),
    ("Stage 2 main sample (gig-proxy vs traditional)", int(stage2["y_gig_vs_trad"].notna().sum())),
], columns=["group", "women"])
display(flow); flow.to_csv(OUT_DIR / "sample_flow.csv", index=False)

split = pd.DataFrame([
    ("Stage 1 participation", 108_412, 27_413, "35.2%", "35.7%"),
    ("Stage 2 main: gig vs traditional", 29_930, 7_516, "5.9%", "6.1%"),
    ("Stage 2 robustness: gig vs all others", 35_574, 8_917, "5.0%", "5.0%"),
    ("Stage 2 sensitivity: gig+startup vs traditional", 35_603, 8_888, "20.7%", "21.2%"),
], columns=["task", "train rows", "test rows", "positive share (train)", "positive share (test)"])
display(split); split.to_csv(OUT_DIR / "train_test_split.csv", index=False)

In [ ]:
#3
stage1["kids_u6_band"] = stage1["hh_kids_u6"].clip(upper=3).astype(int).astype(str).replace({"3": "3+"})
stage1["age_band"] = pd.cut(stage1["age"], [14, 24, 34, 44, 54, 59],
                            labels=["15-24", "25-34", "35-44", "45-54", "55-59"])

def part_table(col):
    key = lab(stage1[col], col) if col in SRC else stage1[col]
    return (stage1.groupby(key, observed=True)["lfp"]
            .agg(participation="mean", n="size").round(3))

tables = {}
for col in ["edu_group", "sector", "social_group", "marital", "religion", "kids_u6_band", "age_band"]:
    tables[col] = part_table(col)
    print(f"\n--- Participation by {col}")
    display(tables[col])

st = part_table("state").sort_values("participation")
print("\n--- Lowest 5 and highest 5 states")
display(pd.concat([st.head(5), st.tail(5)]))

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), ["edu_group", "sector", "social_group", "marital", "kids_u6_band", "age_band"]):
    tables[col]["participation"].plot.bar(ax=ax, color="#1F3A5F")
    ax.set_title(f"Participation by {col}"); ax.set_ylabel("Share in labour force"); ax.set_xlabel("")
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_eda_participation.png", dpi=150); plt.show()

In [ ]:
#3
prof = stage2.groupby("own_economy_type").agg(
    n=("HHID", "size"), mean_age=("age", "mean"), mean_edu_years=("edu_years", "mean"),
    mean_log_pc_exp=("log_pc_exp", "mean"), mean_kids_u6=("hh_kids_u6", "mean"))
prof["rural_share"] = stage2.assign(r=(stage2["sector"] == "1")).groupby("own_economy_type")["r"].mean()
display(prof.round(3))

for col in ["edu_group", "social_group", "marital"]:
    key = lab(stage2[col], col) if col in SRC else stage2[col]
    print(f"\n--- {col}: share of each economy type within each group (row %)")
    display(pd.crosstab(key, stage2["own_economy_type"], normalize="index").round(3))

prof.to_csv(OUT_DIR / "eda_economy_profile.csv")

In [ ]:
#4
work = stage1[stage1["lfp_group"] == "working"]
rows = []
for c in NUM:
    lo, hi = work[c].quantile([0.01, 0.99])
    sd = np.sqrt((work[c].var() + outside[c].var()) / 2)
    rows.append({"feature": c,
                 "mean_working": work[c].mean(), "mean_outside": outside[c].mean(),
                 "std_diff": (outside[c].mean() - work[c].mean()) / sd if sd > 0 else np.nan,
                 "share_outside_beyond_working_1_99pct": ((outside[c] < lo) | (outside[c] > hi)).mean()})
support = pd.DataFrame(rows).round(3)
display(support)
support.to_csv(OUT_DIR / "eda_common_support.csv", index=False)

for c in CAT:
    unseen = set(outside[c].dropna().unique()) - set(work[c].dropna().unique())
    print(c, "- categories among non-participants never seen in working women:", unseen or "none")